# Pacman training on Kaggle

Trains the A2C/PPO or the DQN agent of [CL-BZH/Pacman](https://github.com/CL-BZH/Pacman) on a GPU
(choose with `AGENT` in the configuration cell).

**Before running:** in the notebook settings (right panel) set *Accelerator → GPU T4 x2* (or P100) and turn *Internet* on (needed to clone the repository; it requires a verified phone number).

A Kaggle session lasts up to 12 hours. For long trainings use *Save Version → Save & Run All (Commit)*: the notebook then runs in the background even if you close the browser, and the files in `/kaggle/working` become the version's output (to resume later, add that output as an input with *Add Input* and set `RESUME_FROM = "auto"`, or the path of the checkpoint under `/kaggle/input/...`).

Checkpoints (`pacman_<agent>_<step>.pth`, `pacman_<agent>_final.pth`) and the per-game log
(`pacman_<agent>_training_log.csv`) are written to `SAVE_DIR`. To continue training in a new
session, set `RESUME_FROM = "auto"` (on Kaggle, first add the previous version's output with *Add Input*).

To watch the trained agent play locally, copy the checkpoint into the repository and load it
in `pacman_play.py`.

## 1. Setup

In [ ]:
SAVE_DIR = "/kaggle/working/pacman_rl"
import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import os
REPO_DIR = "/kaggle/working/Pacman"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/CL-BZH/Pacman.git {REPO_DIR}
assert os.path.isdir(REPO_DIR), "Cloning the repository failed: check that Internet is ON in the notebook settings (it requires a verified phone number)."
%cd {REPO_DIR}
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: training will run on CPU"
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Configuration

In [ ]:
# ---- Training configuration ----------------------------------------------
AGENT       = "a2c"           # "a2c" (actor-critic trained with PPO) or "dqn"
TOTAL_STEPS = 20_000_000       # environment steps for this session (all envs together)
N_ENVS      = 16
N_FRAMES    = 2               # must match the checkpoint when resuming
SAVE_EVERY  = 1_000_000       # checkpoint interval (steps)

# To continue a previous session:
#   RESUME_FROM = "auto"   -> the most recent checkpoint of AGENT, found automatically
#                             (Kaggle: first add the previous version's output with
#                              "Add Input" -> Notebooks -> this notebook)
#   RESUME_FROM = "/path/to/checkpoint.pth"
RESUME_FROM = None

## 3. Training

In [ ]:
from pacman_training import pacman_conf
from pacman_env import PacmanEnvironment
from pacman_agents import A2C_PacmanAgent, DQN_PacmanAgent

def make_agent(envs):
    if AGENT == "a2c":
        return A2C_PacmanAgent(envs, n_frames=N_FRAMES)
    return DQN_PacmanAgent(envs, n_frames=N_FRAMES, buffer_capacity=300_000)

import glob, os, re

def find_checkpoint(resume_from):
    """
    Path of the checkpoint to resume from.
    "auto": the most recent checkpoint of AGENT found in the notebook's inputs (/kaggle/input,
    a previous version's output added with "Add Input") or in SAVE_DIR:
    pacman_<agent>_final.pth if there is one, else the pacman_<agent>_<step>.pth with the largest step.
    """
    if resume_from is None:
        return None
    if resume_from != "auto":
        if not os.path.isfile(resume_from):
            raise FileNotFoundError(f"{resume_from} not found. Checkpoints found: "
                                    f"{glob.glob('/kaggle/input/**/*.pth', recursive=True) + glob.glob(SAVE_DIR + '/*.pth')}")
        return resume_from
    found = glob.glob(f"/kaggle/input/**/pacman_{AGENT}_*.pth", recursive=True) + \
            glob.glob(f"{SAVE_DIR}/pacman_{AGENT}_*.pth")
    def order(path):
        name = os.path.basename(path)
        if name == f"pacman_{AGENT}_final.pth":
            return float("inf")
        m = re.match(rf"pacman_{AGENT}_(\d+)\.pth$", name)
        return int(m.group(1)) if m else -1
    found = [p for p in found if order(p) >= 0]
    if not found:
        raise FileNotFoundError(
            f"No checkpoint pacman_{AGENT}_*.pth found in /kaggle/input or {SAVE_DIR}. On Kaggle, "
            "each version starts with an empty /kaggle/working: add the output of the previous "
            "version with 'Add Input' (Notebooks -> this notebook).")
    return max(found, key=order)

checkpoint = find_checkpoint(RESUME_FROM)
resuming = checkpoint is not None
agent = make_agent([PacmanEnvironment(pacman_conf) for _ in range(N_ENVS)])
if resuming:
    agent.load(checkpoint)
    print("Resumed from", checkpoint)

if AGENT == "a2c":
    agent.A2C.set_lr(1e-4 if resuming else 2.5e-4)
    agent.lr = 1e-4 if resuming else 2.5e-4
    history = agent.run_episodes(
        total_steps=TOTAL_STEPS, n_steps=128, episode_max_length=1000, min_lr=2.5e-5,
        # Curriculum on the ghosts' aggressiveness (not when resuming a trained agent)
        curriculum_fraction=0. if resuming else 0.2, curriculum_start=0.3,
        log_interval=1000, save_interval=SAVE_EVERY, save_dir=SAVE_DIR)
else:
    if resuming:
        agent.DQN.set_lr(1e-4)
        agent.lr = 1e-4
    history = agent.run_episodes(
        total_steps=TOTAL_STEPS, episode_max_length=1000,
        # When resuming, the policy is already trained: explore much less
        epsilon_start=0.10 if resuming else 1.0, epsilon_end=0.02,
        epsilon_fraction=0.05 if resuming else 0.25,
        learning_starts=20_000, train_freq=1, gradient_steps=1, min_lr=5e-5,
        log_interval=1000, save_interval=SAVE_EVERY, save_dir=SAVE_DIR)

## 4. Training curves

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

log = pd.read_csv(f"{SAVE_DIR}/pacman_{AGENT}_training_log.csv")
w = 2000
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(log["step"], log["reward"].rolling(w).mean()); ax[0].set_title("Reward per game")
ax[1].plot(log["step"], 100 * log["won"].rolling(w).mean(), label="won")
ax[1].plot(log["step"], 100 * log["lost"].rolling(w).mean(), label="lost")
ax[1].set_title("Games (%)"); ax[1].legend()
ax[2].plot(log["step"], log["ghosts_left"].rolling(w).mean()); ax[2].set_title("Ghosts left at the end")
for a in ax: a.set_xlabel("environment steps")
plt.tight_layout(); plt.show()

## 5. Evaluation

In [ ]:
def evaluate(checkpoint, n_games=1000, max_moves=800):
    """Play n_games with the configured ghosts' aggressiveness, greedy policy when ghosts are present."""
    env = PacmanEnvironment(pacman_conf)
    player = make_agent([env])
    player.load(checkpoint)
    if AGENT == "dqn":
        player.epsilon = 0.
    won = lost = aborted = 0
    for _ in range(n_games):
        player.reset_episode()
        state = player.current_state[0]
        moves = 0
        while not env.end and moves < max_moves:
            # Like pacman_play.py: stochastic policy for A2C once all ghosts are eaten
            _, action = player.select_action(state, stockastic=(AGENT == "a2c" and env.ghosts == []))
            env.step(action)
            state = player.observe(0)
            moves += 1
        won += env.end and not env.dead
        lost += env.dead
        aborted += not env.end
    print(f"{checkpoint}: won {100*won/n_games:.1f}%, lost {100*lost/n_games:.1f}%, "
          f"aborted {100*aborted/n_games:.1f}%")

evaluate(f"{SAVE_DIR}/pacman_{AGENT}_final.pth")